<a href="https://colab.research.google.com/github/hUSsAin976-tech/ML-internship-at-FlyRank/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

Lane: **AI Referral Opportunity** (freestyle). This section turns the validated ML-08/ML-09 model
into the actual deliverable: a ranked queue over the real opportunity population (demand-worthy
pages with zero AI-referred sessions this month), with a reason code, an archetype, a confidence
tier, and a specific action per row.

Same data contract as ML-04/ML-07/ML-08/ML-09: `fact_content_daily_performance`, `month=2026-03`
partition only (never `_sample`), joined to `dim_content`. Label stays `has_ai_sessions_month` —
an evidence variable, not a forecast target.

> Working with an AI assistant? Read `skills/README.md`, then load `writing-honest-claims` +
> `flyrank/flyrank-data` for this task.


In [1]:
import os, getpass
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_content":      f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily_month": f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
}

# Identical content_month view to ML-07/ML-08/ML-09 -- same data contract, same features, so this
# playbook scores the SAME population and columns already validated, nothing new smuggled in at
# the last step.
con.sql(f'''
    CREATE OR REPLACE TEMP VIEW content_month AS
    SELECT
        f.content_hash_id,
        ANY_VALUE(f.client_hash_id)                                                  AS client_hash_id,
        SUM(f.gsc_impressions)                                                       AS total_gsc_impressions_month,
        AVG(CASE WHEN f.gsc_impressions > 0 THEN f.gsc_avg_position END)             AS avg_gsc_position_month,
        COUNT(DISTINCT CASE WHEN f.gsc_impressions > 0 THEN f.report_date END)       AS days_with_impressions_month,
        MAX(CASE WHEN f.ga4_data_available IS TRUE AND f.sessions_ai > 0
                 THEN 1 ELSE 0 END)                                                  AS has_ai_sessions_month
    FROM {TABLES['fact_daily_month']} f
    WHERE f.ga4_data_available IS TRUE
    GROUP BY 1
''')

raw = con.sql(f'''
    SELECT
        cm.content_hash_id,
        cm.client_hash_id,
        cm.total_gsc_impressions_month,
        cm.avg_gsc_position_month,
        cm.days_with_impressions_month,
        cm.has_ai_sessions_month,
        dc.word_count,
        dc.content_type,
        DATE_DIFF('day', dc.content_updated_date, DATE '2026-03-31') AS days_since_update_month
    FROM content_month cm
    JOIN {TABLES['dim_content']} dc USING (content_hash_id)
''').df()
raw["days_since_update_month"] = raw["days_since_update_month"].clip(lower=0)

MIN_DEMAND_IMPRESSIONS = 100

# Labeled population -- used to FIT the scorer. Same demand-worthy slice ML-08/ML-09 validated on
# (impressions >= 100, both positives and negatives present -- lift@K needs positives to measure
# anything).
labeled = raw[raw["total_gsc_impressions_month"] >= MIN_DEMAND_IMPRESSIONS].copy()

# Action-queue population -- the actual opportunity pool this playbook ranks and delivers.
# Demand-worthy AND zero AI-referred sessions this month, identical to ML-07's eligible pool. By
# construction this pool has NO positives (that's the eligibility rule), so it is scored by the
# model fit on `labeled` above, never validated against itself.
eligible = raw[
    (raw["total_gsc_impressions_month"] >= MIN_DEMAND_IMPRESSIONS)
    & (raw["has_ai_sessions_month"] == 0)
].copy()

print(f"labeled (fit) population: {len(labeled):,} rows, {labeled['client_hash_id'].nunique()} clients, "
      f"base rate {labeled['has_ai_sessions_month'].mean():.2%}")
print(f"action-queue (opportunity) population: {len(eligible):,} of {len(raw):,} content items "
      f"({len(eligible)/len(raw):.1%}) -- demand-worthy, zero AI sessions this month")

Paste your Hugging Face READ token (hf_...): ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

labeled (fit) population: 32,596 rows, 30 clients, base rate 8.61%
action-queue (opportunity) population: 29,788 of 90,489 content items (32.9%) -- demand-worthy, zero AI sessions this month


**Why refit on all labeled rows, not just ML-08's training split.** ML-08/ML-09 already
answered the question this playbook needs answered before it can trust a score: does the fitted
score generalize to clients the model never saw? Grouped-split validation said yes (Logistic
Regression: lift@K 5.02x vs the baseline's 3.01x, on clients held out entirely — see
`w06_validation_audit.ipynb` Section 2). That question doesn't need re-asking every time the
score is deployed. For the queue below, the same architecture (Logistic Regression, same five
features + freshness + missingness flags, `class_weight="balanced"`) is refit on the full labeled
population so it sees every available example before scoring the actual opportunity pool — this
is standard deployment practice once generalization is validated, not a new, unvalidated model.

**Archetype → action mapping.** Four signals, already validated in ML-07/ML-08/ML-09, combine
into an archetype for each row:

| Signal | Source | What it means |
|---|---|---|
| `already_page1` | `avg_gsc_position_month <= 10` | ML-07's own top-20 read found most high-scoring picks already rank on page 1 — "AI can't find it" is a weaker explanation here than for a page that isn't visible in search at all |
| `type_affinity` | sign of this fit's Logistic Regression coefficient for `content_type` | ML-08 found `feedly article` carries the strongest *positive* AI-referral coefficient and `keyword article` the strongest *negative* one — a content type's typical AI-referral behavior, not a page-level fact |
| `confidence` | agreement between the model's top-K and the baseline's (volume-only) top-K | ML-08 Section 4's disagreement analysis, extended into a three-way tier |
| `staleness (context only)` | `days_since_update_month` | ML-07 Signal B came back **MIXED** (non-monotonic, near-zero correlation) — staleness is carried through as a reported column for human context, never as a scoring input or a reason code |

| Archetype | Condition | Action |
|---|---|---|
| `page1_high_affinity_gap` | already page 1 AND a high-affinity content type | `priority_ai_answerability_review` |
| `page1_zero_ai_referral` | already page 1, any other content type | `ai_answerability_review` |
| `underperforming_high_affinity_type` | not page 1, but a type that normally does well for AI referral | `content_quality_review` |
| `low_affinity_content_type` | not page 1, a type that normally does poorly | `format_restructure_review` |
| `general_opportunity` | none of the above | `standard_ai_visibility_review` |

Confidence (`high` / `medium` / `low`) is reported **alongside** the archetype, not folded into
it — it changes how much a human should trust the row, not what kind of fix it needs.


In [2]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

NUMERIC_FEATURES = [
    "total_gsc_impressions_month", "avg_gsc_position_month",
    "days_with_impressions_month", "word_count", "days_since_update_month",
]
CATEGORICAL_FEATURES = ["content_type"]
TARGET = "has_ai_sessions_month"

def prep(frame):
    # Explicit has_-flags before fillna, per the flyrank-data skill's missingness gotcha --
    # missingness follows content_type, so a blind fillna(0) would silently encode a
    # content-type signal into the imputed value.
    out = frame.copy()
    for c in ["word_count", "avg_gsc_position_month", "days_since_update_month"]:
        out[f"has_{c}"] = out[c].notna().astype(int)
        out[c] = out[c].astype(float).fillna(out[c].median())
    return out

FLAG_FEATURES = [f"has_{c}" for c in ["word_count", "avg_gsc_position_month", "days_since_update_month"]]
ALL_NUMERIC = NUMERIC_FEATURES + FLAG_FEATURES

labeled_p  = prep(labeled)
eligible_p = prep(eligible)

preprocess = ColumnTransformer([
    ("num", StandardScaler(), ALL_NUMERIC),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
])

final_model = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
]).fit(labeled_p[ALL_NUMERIC + CATEGORICAL_FEATURES], labeled_p[TARGET])

# Read the content_type affinity straight off THIS fit's own coefficients -- not hardcoded from
# ML-08's numbers, so this stays correct even if the underlying data shifts.
feature_names = (
    ALL_NUMERIC
    + list(final_model.named_steps["prep"].named_transformers_["cat"].get_feature_names_out(CATEGORICAL_FEATURES))
)
coefs = pd.Series(final_model.named_steps["clf"].coef_[0], index=feature_names)
type_coefs = {c.replace("content_type_", ""): coefs[c] for c in coefs.index if c.startswith("content_type_")}
HIGH_AFFINITY_TYPES = {t for t, v in type_coefs.items() if v > 0.25}
LOW_AFFINITY_TYPES  = {t for t, v in type_coefs.items() if v < -0.25}

print("content_type affinity read off this fit's coefficients:")
print(pd.Series(type_coefs).sort_values(ascending=False).round(3).to_string())
print(f"\nhigh-affinity types: {HIGH_AFFINITY_TYPES or '(none clear)'}")
print(f"low-affinity types:  {LOW_AFFINITY_TYPES or '(none clear)'}")

model_score    = final_model.predict_proba(eligible_p[ALL_NUMERIC + CATEGORICAL_FEATURES])[:, 1]
baseline_score = eligible_p["total_gsc_impressions_month"].rank(pct=True).values

eligible_p["model_score"]    = model_score
eligible_p["baseline_score"] = baseline_score

K = max(50, int(round(0.05 * len(eligible_p))))
model_topk_idx    = eligible_p["model_score"].rank(ascending=False, method="first") <= K
baseline_topk_idx = eligible_p["baseline_score"].rank(ascending=False, method="first") <= K

def confidence_tier(in_model, in_baseline):
    if in_model and in_baseline:
        return "high"
    if in_model or in_baseline:
        return "medium"
    return "low"

eligible_p["confidence"] = [
    confidence_tier(m, b) for m, b in zip(model_topk_idx, baseline_topk_idx)
]

def assign_archetype(row):
    page1 = pd.notna(row["avg_gsc_position_month"]) and row["avg_gsc_position_month"] <= 10
    high_aff = row["content_type"] in HIGH_AFFINITY_TYPES
    low_aff  = row["content_type"] in LOW_AFFINITY_TYPES
    if page1 and high_aff:
        return "page1_high_affinity_gap", "priority_ai_answerability_review"
    if page1:
        return "page1_zero_ai_referral", "ai_answerability_review"
    if high_aff:
        return "underperforming_high_affinity_type", "content_quality_review"
    if low_aff:
        return "low_affinity_content_type", "format_restructure_review"
    return "general_opportunity", "standard_ai_visibility_review"

archetypes, actions = zip(*eligible_p.apply(assign_archetype, axis=1))
eligible_p["archetype"] = archetypes
eligible_p["action"] = actions

def reason_codes(row):
    codes = ["high_demand_zero_ai_sessions"]
    if row["archetype"] in ("page1_high_affinity_gap", "page1_zero_ai_referral"):
        codes.append("already_page1_zero_ai")
    if row["archetype"] in ("page1_high_affinity_gap", "underperforming_high_affinity_type"):
        codes.append("high_affinity_type_underperforming")
    if row["archetype"] == "low_affinity_content_type":
        codes.append("low_affinity_type")
    codes.append(f"confidence_{row['confidence']}")
    return "|".join(codes)

eligible_p["reason_code"] = eligible_p.apply(reason_codes, axis=1)

ranked_queue = eligible_p.sort_values("model_score", ascending=False).reset_index(drop=True)
ranked_queue["rank"] = np.arange(1, len(ranked_queue) + 1)

queue_cols = [
    "rank", "content_hash_id", "client_hash_id", "model_score", "baseline_score",
    "archetype", "action", "confidence", "reason_code",
    "total_gsc_impressions_month", "avg_gsc_position_month", "days_with_impressions_month",
    "days_since_update_month", "word_count", "content_type",
]
ranked_queue = ranked_queue[queue_cols]

print(f"\nranked queue: {len(ranked_queue):,} rows")
print(f"K (top-5%-or-50) used for confidence tiering: {K}")
print("\narchetype distribution:")
print(ranked_queue["archetype"].value_counts().to_string())
print("\nconfidence distribution:")
print(ranked_queue["confidence"].value_counts().reindex(["high", "medium", "low"]).to_string())
ranked_queue.head(20)

content_type affinity read off this fit's coefficients:
feedly article        1.510
comparison article    0.005
keyword article      -1.039

high-affinity types: {'feedly article'}
low-affinity types:  {'keyword article'}

ranked queue: 29,788 rows
K (top-5%-or-50) used for confidence tiering: 1489

archetype distribution:
archetype
page1_zero_ai_referral                17669
low_affinity_content_type             11959
page1_high_affinity_gap                 131
underperforming_high_affinity_type       24
general_opportunity                       5

confidence distribution:
confidence
high        648
medium     1682
low       27458


,rank,content_hash_id,client_hash_id,model_score,baseline_score,archetype,action,confidence,reason_code,total_gsc_impressions_month,avg_gsc_position_month,days_with_impressions_month,days_since_update_month,word_count,content_type
0,1,content_eadb33b5df496f4a,client_e547b89c05043229,0.999694,1.000000,page1_zero_ai_referral,ai_answerability_review,high,high_demand_zero_ai_sessions|already_page1_zer...,617124.0,2.383011,29,0.0,2753.0,keyword article
1,2,content_ec2e0346994fb5a5,client_e547b89c05043229,0.982326,0.999966,page1_zero_ai_referral,ai_answerability_review,high,high_demand_zero_ai_sessions|already_page1_zer...,245276.0,2.854514,29,0.0,2581.0,keyword article
2,3,content_fa84f5976d5fe3c1,client_23a62021009f63c4,0.979934,0.999396,low_affinity_content_type,format_restructure_review,high,high_demand_zero_ai_sessions|low_affinity_type...,78716.0,36.535261,31,0.0,3627.0,keyword article
3,4,content_0e03de7680314cd5,client_e547b89c05043229,0.977566,0.999933,page1_zero_ai_referral,ai_answerability_review,high,high_demand_zero_ai_sessions|already_page1_zer...,221310.0,2.675217,29,0.0,2784.0,keyword article
4,5,content_7ea10d9117b8b4fa,client_23a62021009f63c4,0.977499,0.994226,low_affinity_content_type,format_restructure_review,high,high_demand_zero_ai_sessions|low_affinity_type...,29551.0,37.951278,31,34.0,6488.0,keyword article
5,6,content_d6fe89a595f07242,client_b10cb2997d0c7c86,0.976138,0.878743,page1_high_affinity_gap,priority_ai_answerability_review,medium,high_demand_zero_ai_sessions|already_page1_zer...,4187.0,7.363824,27,34.0,1387.0,feedly article
6,7,content_df47d1b976106de4,client_23a62021009f63c4,0.974411,0.999799,low_affinity_content_type,format_restructure_review,high,high_demand_zero_ai_sessions|low_affinity_type...,124727.0,24.123242,29,0.0,3435.0,keyword article
7,8,content_0c40fae0b03c27e6,client_23a62021009f63c4,0.970575,0.980848,low_affinity_content_type,format_restructure_review,high,high_demand_zero_ai_sessions|low_affinity_type...,16035.0,35.833779,31,34.0,6138.0,keyword article
8,9,content_5941f92782343e72,client_20259bd6705d81d4,0.969085,0.999228,low_affinity_content_type,format_restructure_review,high,high_demand_zero_ai_sessions|low_affinity_type...,69960.0,33.465033,29,0.0,3743.0,keyword article
9,10,content_4ffe18112a5642e3,client_e547b89c05043229,0.968488,0.999899,page1_zero_ai_referral,ai_answerability_review,high,high_demand_zero_ai_sessions|already_page1_zer...,186983.0,2.331060,29,0.0,3097.0,keyword article


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended use.** This queue is a **prioritization aid** for a content or SEO team deciding which
pages to look at first for AI-referral opportunity. It ranks pages by a validated resemblance
score (fit on this month's warehouse partition, grouped-split tested in
`w06_validation_audit.ipynb`) — it does not diagnose *why* a page gets no AI-referred sessions,
and it does not guarantee that acting on any single row will produce AI referral. Per
`writing-honest-claims`, the correct frame is **decision-support**: "these pages look worth
reviewing first, because…", never "fixing this page will get it cited."

**Scope this queue is valid for** (numbers behind each line in the cell below):
- One warehouse partition, `month=2026-03` — a single snapshot, not a trend, not a forecast.
- Content items with real, current search demand (`total_gsc_impressions_month >= 100`) and GA4
  tracking available for that client (`ga4_data_available = TRUE`) — pages outside this filter
  were never scored and should not be assumed similar.
- The client mix present in this partition. The grouped-split test showed the score holds on
  clients unseen during fitting, but it was tested on roughly 30 clients total — a genuinely new
  client outside that distribution (different vertical, different site template) is untested
  territory.

**Where it stops being valid:**
- **Cross-sectional, not causal.** The model was fit on an observed association between content
  signals and `has_ai_sessions_month`, not on any experiment. It cannot tell you that reviewing a
  page *will* produce AI referral — only that pages like it, in this data, more often show
  AI-referred sessions.
- **`has_ai_sessions_month` is an evidence variable, not a forecast target** (per the ML-04 data
  contract, carried through ML-07/ML-08/ML-09) — a zero this month does not mean "will never
  happen," and a model flag does not mean "will happen next month."
- **Small buckets, thin clients.** Some archetypes and some clients contribute very few rows to
  the queue (printed below) — treat single-digit-n slices as directional, not decisive.
- **Not a substitute for GA4 auditing.** A `has_ai_sessions_month = 0` reading can reflect a real
  absence of AI referral *or* a GA4/tracking gap for that specific client. The model has no way
  to tell these apart from the columns available to it.


In [3]:
n_total = len(raw)
n_labeled = len(labeled)
n_eligible = len(eligible)
n_clients_eligible = ranked_queue["client_hash_id"].nunique()

print(f"total content items scored this month (GA4-available): {n_total:,}")
print(f"labeled (fit) population: {n_labeled:,} ({n_labeled/n_total:.1%} of total)")
print(f"delivered action queue: {n_eligible:,} ({n_eligible/n_total:.1%} of total)")
print(f"clients represented in the queue: {n_clients_eligible}")

per_client = ranked_queue["client_hash_id"].value_counts()
print("\nrows per client in the delivered queue (thin-client check):")
print(per_client.describe()[["min", "25%", "50%", "75%", "max"]].round(1).to_string())
thin_clients = (per_client < 5).sum()
print(f"\nclients contributing fewer than 5 rows to the queue: {thin_clients} of {len(per_client)}")

total content items scored this month (GA4-available): 90,489
labeled (fit) population: 32,596 (36.0% of total)
delivered action queue: 29,788 (32.9% of total)
clients represented in the queue: 30

rows per client in the delivered queue (thin-client check):
min       1.0
25%      16.8
50%     136.5
75%     580.2
max    7746.0

clients contributing fewer than 5 rows to the queue: 4 of 30


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Cost/value framing, before the rules.** A false positive here costs an analyst a few minutes
reading a page that turns out fine — cheap. A false negative (a real opportunity that never
enters the queue at all, because it fell below the `impressions >= 100` demand floor or the
client has no GA4 tracking) costs nothing *visible*, which is the more dangerous failure mode: it
never shows up to be caught. That asymmetry is why review effort below is weighted toward the
**low-confidence and edge-of-scope rows**, not spread evenly — a `high`-confidence,
`page1_high_affinity_gap` row is cheap to trust; a `low`-confidence row from a thin client is
where a person's ten minutes buys the most protection against acting on noise.

**A human must check, before any row leads to a real content change:**
1. **The zero-AI-session reading itself** — pull the client's GA4 property and confirm
   AI-referral tracking is actually live for this page, not silently broken (Section 2's
   tracking-gap caveat).
2. **The archetype's premise** — for `page1_*` archetypes, confirm the page really is ranking
   well today (SERP position drifts daily; `avg_gsc_position_month` is a monthly average).
3. **Client/business context** the model has no access to — a page slated for removal, a
   client-specific embargo, a page already mid-rewrite.
4. **Low-confidence rows specifically** — these are cases where the validated model and the
   simple volume rule disagree; read the row, don't just trust the rank.

**What should NOT be automated:**
- **No auto-publishing or auto-editing.** This queue produces a reading list, never a content
  change.
- **No client-facing guarantees.** Never present a row's presence in this queue, or its
  `model_score`, as a promise of future AI citation — see Section 2's causal-limit note.
- **No cross-month or cross-client blending without re-validation.** The score is fit and scoped
  to `month=2026-03`; feeding it a different month or a client outside this partition without
  re-checking the grouped split is exactly the mistake ML-09 audited against.
- **No skipping the GA4-tracking check for `high`-confidence rows.** Confidence describes
  model/baseline agreement, not tracking reliability — the two are independent risks.


In [4]:
def needs_human_review(row):
    reasons = []
    if row["confidence"] == "low":
        reasons.append("low_confidence")
    if per_client.get(row["client_hash_id"], 0) < 5:
        reasons.append("thin_client_n")
    if row["content_type"] not in set(labeled["content_type"].dropna().unique()):
        reasons.append("unseen_content_type")
    if pd.isna(row["avg_gsc_position_month"]):
        reasons.append("no_position_data")
    return "|".join(reasons) if reasons else "none"

ranked_queue["human_review_flags"] = ranked_queue.apply(needs_human_review, axis=1)
ranked_queue["human_review_required"] = ranked_queue["human_review_flags"] != "none"

print(f"rows flagged for mandatory human review before acting: "
      f"{ranked_queue['human_review_required'].sum():,} of {len(ranked_queue):,} "
      f"({ranked_queue['human_review_required'].mean():.1%})")
print("\nreview-flag breakdown:")
print(ranked_queue.loc[ranked_queue["human_review_required"], "human_review_flags"]
      .str.split("|").explode().value_counts().to_string())

rows flagged for mandatory human review before acting: 27,459 of 29,788 (92.2%)

review-flag breakdown:
human_review_flags
low_confidence    27458
thin_client_n         7


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

This month's numbers (printed below) are the **reference reading**. None of this is live
monitoring — the warehouse ships one partition at a time, and there's no second month scored yet
to compare against — so these are the concrete thresholds to check the next time a new partition
is available, not a dashboard that exists today.

**Retrain / re-review triggers:**
- **Base rate drift.** If `has_ai_sessions_month`'s base rate in a new labeled partition moves by
  more than roughly 2x from this month's reading (printed below), the model's calibration should
  be re-checked before trusting its scores on the new month.
- **Lift@K decay.** If a fresh grouped-split validation (same recipe as
  `w06_validation_audit.ipynb`) shows lift@K dropping toward the baseline's own lift (i.e. the
  model stops beating the simple volume rule), retrain or fall back to the baseline rule — the
  model only earns its place in the queue by beating that bar.
- **Archetype mix shift.** A large swing in the `archetype` distribution (printed below) between
  partitions — e.g. `page1_*` archetypes suddenly dominating — signals the underlying content mix
  changed and the archetype thresholds may need revisiting, not just the model weights.
- **New/changed content types.** If `content_type` values appear that weren't in this month's
  `labeled` population, the affinity split (`HIGH_AFFINITY_TYPES` / `LOW_AFFINITY_TYPES`) has no
  evidence for them — those rows default to `general_opportunity` and pick up a mandatory review
  flag (Section 3) rather than a silent guess.
- **Client roster change.** If more than roughly a third of clients in a new partition weren't
  part of this month's fit population, the grouped-split generalization claim from ML-09 hasn't
  been tested on them — re-validate before trusting the queue for those clients specifically.

**What would NOT justify a retrain on its own:** a single low-confidence row looking wrong on
manual read — Section 3 already expects some of those; that's what the review layer is for, not
a trigger to change the model.


In [5]:
print("--- Reference reading: month=2026-03 (compare future partitions against this) ---")
print(f"labeled population base rate: {labeled['has_ai_sessions_month'].mean():.2%}")
print(f"delivered queue size: {len(ranked_queue):,}")
print(f"content_type affinity this fit: high={sorted(HIGH_AFFINITY_TYPES)}, low={sorted(LOW_AFFINITY_TYPES)}")
print("\narchetype mix (reference distribution):")
print((ranked_queue["archetype"].value_counts(normalize=True) * 100).round(1).astype(str).add("%").to_string())
print("\nvalidated model performance to compare future partitions against (from w06_validation_audit.ipynb):")
print("  baseline (volume-rank) lift@K: 3.01x")
print("  Logistic Regression lift@K (grouped, honest split): 5.02x")
print("  -> re-validate if a fresh grouped-split lift@K falls notably below ~5x, "
      "or approaches the baseline's ~3x.")

--- Reference reading: month=2026-03 (compare future partitions against this) ---
labeled population base rate: 8.61%
delivered queue size: 29,788
content_type affinity this fit: high=['feedly article'], low=['keyword article']

archetype mix (reference distribution):
archetype
page1_zero_ai_referral                59.3%
low_affinity_content_type             40.1%
page1_high_affinity_gap                0.4%
underperforming_high_affinity_type     0.1%
general_opportunity                    0.0%

validated model performance to compare future partitions against (from w06_validation_audit.ipynb):
  baseline (volume-rank) lift@K: 3.01x
  Logistic Regression lift@K (grouped, honest split): 5.02x
  -> re-validate if a fresh grouped-split lift@K falls notably below ~5x, or approaches the baseline's ~3x.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on
these files.*

The queue CSV is intentionally gitignored (`work/**/*.csv`, the leak-guard) — this notebook
regenerates it. The metrics JSON and the two figures below are small and get committed; they're
the receipts next week's paper points back to.


In [6]:
import json
import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT_DIR = "../outputs"
FIG_DIR = "../figures"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# 1. The ranked queue itself -- gitignored by design (work/**/*.csv); this notebook regenerates it.
queue_path = os.path.join(OUT_DIR, "w07_action_playbook_queue.csv")
ranked_queue.to_csv(queue_path, index=False)
print(f"wrote {len(ranked_queue):,} rows to {queue_path}")

# 2. Metrics JSON -- small, committed, the numeric receipts the paper's numbers trace back to.
metrics = {
    "month_partition": "2026-03",
    "labeled_population_rows": int(len(labeled)),
    "labeled_population_base_rate": float(labeled["has_ai_sessions_month"].mean()),
    "delivered_queue_rows": int(len(ranked_queue)),
    "delivered_queue_clients": int(ranked_queue["client_hash_id"].nunique()),
    "confidence_tiering_K": int(K),
    "archetype_counts": ranked_queue["archetype"].value_counts().to_dict(),
    "confidence_counts": ranked_queue["confidence"].value_counts().to_dict(),
    "human_review_required_rows": int(ranked_queue["human_review_required"].sum()),
    "content_type_affinity": {
        "high": sorted(HIGH_AFFINITY_TYPES),
        "low": sorted(LOW_AFFINITY_TYPES),
    },
    "validated_lift_at_k_reference": {
        "source_notebook": "w06_validation_audit.ipynb",
        "baseline_volume_rank": 3.01,
        "logistic_regression_grouped_split": 5.02,
    },
}
metrics_path = os.path.join(OUT_DIR, "w07_action_playbook_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2, default=str)
print(f"wrote metrics to {metrics_path}")

# 3. Two figures for the paper -- committed to work/figures/ (not gitignored).
archetype_counts = ranked_queue["archetype"].value_counts()
plt.figure(figsize=(8, 4))
archetype_counts.plot(kind="barh", color="#426B69")
plt.title("Action queue by archetype")
plt.xlabel("content items")
plt.tight_layout()
archetype_fig_path = os.path.join(FIG_DIR, "w07_archetype_mix.svg")
plt.savefig(archetype_fig_path)
plt.close()
print(f"wrote {archetype_fig_path}")

confidence_counts = ranked_queue["confidence"].value_counts().reindex(["high", "medium", "low"])
plt.figure(figsize=(6, 4))
confidence_counts.plot(kind="bar", color="#6F4E7C")
plt.title("Action queue confidence mix")
plt.ylabel("content items")
plt.tight_layout()
confidence_fig_path = os.path.join(FIG_DIR, "w07_confidence_mix.svg")
plt.savefig(confidence_fig_path)
plt.close()
print(f"wrote {confidence_fig_path}")

wrote 29,788 rows to ../outputs/w07_action_playbook_queue.csv
wrote metrics to ../outputs/w07_action_playbook_metrics.json
wrote ../figures/w07_archetype_mix.svg
wrote ../figures/w07_confidence_mix.svg


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.